# 07 — Real galaxy fit (JWST NIRCam F150W)

Loads `data/raw/TEST_F150W_NIRCAM.fits` (a JWST NIRCam Short-Wavelength F150W postage stamp) and fits a single-Sérsic + PSF model.

In [ ]:
# Bootstrap: make `lensing` importable when running notebooks/ directly.
import sys
from pathlib import Path
repo = Path.cwd().resolve().parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import lensing as gl
# Device-agnostic: prefer MPS (Apple GPU) → CUDA → CPU.
# Pass device="cpu" if you need to force the CPU path (e.g. for
# operators that have no MPS kernel yet, or for reproducibility).
device, dtype = gl.config.setup(seed=42)
print(f"using device: {device}")


In [ ]:
from astropy.io import fits

FITS_PATH = repo / 'data' / 'raw' / 'TEST_F150W_NIRCAM.fits'
DELTAPIX  = 0.031   # NIRCam SW pixel scale [arcsec/pix]
PSF_FWHM  = 0.05    # F150W diffraction-limited FWHM [arcsec]

with fits.open(FITS_PATH) as hdul:
    data = np.asarray(hdul[0].data, dtype=np.float32)
data = np.nan_to_num(data)
data = data - np.median(data)
data = torch.tensor(data)
ny, nx = data.shape
print(f"image shape: {data.shape}, max = {float(data.max()):.3e}")

# Square crop of the central source so the model fit doesn't have to
# explain hundreds of background sources.
half_size = 64  # pixels
cy, cx = ny // 2, nx // 2
crop = data[cy-half_size:cy+half_size, cx-half_size:cx+half_size].clone()
ny_c, nx_c = crop.shape
extent = (-nx_c*DELTAPIX/2, nx_c*DELTAPIX/2, -ny_c*DELTAPIX/2, ny_c*DELTAPIX/2)

xy = gl.data.coordinate_grid(npix=nx_c, deltapix=DELTAPIX)

gl.viz.imshow_log(crop + abs(crop.min()) + 1e-3,
                  extent=extent, title='JWST NIRCam F150W (cropped)')
plt.show()


## Single-Sérsic fit with PSF

In [ ]:
sigma_pix = float(crop.std()) * 0.5

res, summary = gl.inference.fit_ellipticity(
    crop, xy, psf_fwhm=PSF_FWHM, deltapix=DELTAPIX, sigma=sigma_pix,
    init=dict(Ie=float(crop.max())*0.5, Re=0.3, n=2.0,
              x0=0., y0=0., e1=0., e2=0.),
    epochs=3000, lr=0.05,
)
print(f'final chi2/dof = {res.best_loss:.3f}')
for k, v in summary.items(): print(f'  {k:<8s}: {v:+.4g}')


In [ ]:
with torch.no_grad():
    model_image = res.model(xy)
shifted = crop + abs(crop.min()) + 1e-3
gl.viz.side_by_side([shifted, model_image, crop - model_image],
                    titles=['data', 'best fit', 'residual'],
                    log=False, extent=extent)
plt.show()
